<a href="https://colab.research.google.com/github/kempegowda7/AI/blob/main/week2_1bm24cs136_DLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from collections import deque

class EightPuzzleDLS:
    def __init__(self, initial_state, final_state):
        # Convert list of lists to tuple of tuples for hashability if needed
        self.initial_state = tuple(tuple(row) for row in initial_state) if isinstance(initial_state[0], list) else initial_state
        self.final_state = tuple(tuple(row) for row in final_state) if isinstance(final_state[0], list) else final_state

    def find_blank(self, state):
        # Find the position (row, col) of the blank space (0) in the board tuple
        for r in range(3):
            for c in range(3):
                if state[r][c] == 0:
                    return r, c
        return -1, -1

    def get_neighbors(self, state):
        # Generates all possible next states from the current board state
        r_blank, c_blank = self.find_blank(state)
        moves = []

        # Possible directions: (dr, dc) for (up, down, left, right)
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        for dr, dc in directions:
            new_r, new_c = r_blank + dr, c_blank + dc

            if 0 <= new_r < 3 and 0 <= new_c < 3:
                # Convert tuple of tuples to list of lists to allow modification
                new_board_list = [list(row) for row in state]
                # Swap blank with the tile at (new_r, new_c)
                new_board_list[r_blank][c_blank], new_board_list[new_r][new_c] = \
                    new_board_list[new_r][new_c], new_board_list[r_blank][c_blank]
                moves.append(tuple(tuple(row) for row in new_board_list)) # Convert back to tuple of tuples
        return moves

    def dls(self, depth_limit):
        # Performs Depth-Limited Search
        # Stack stores (current_state, path_to_current_state, current_depth)
        stack = deque([(self.initial_state, [self.initial_state], 0)])

        # Use a set to keep track of visited states at a given depth
        # (though for DLS, we might revisit states if a shallower path exists from a different branch).
        # A simple 'visited' set can prevent cycles within a single path.
        visited = set()
        visited.add(self.initial_state)

        while stack:
            current_state, path, current_depth = stack.pop()

            if current_state == self.final_state:
                return path, current_depth

            # If depth limit is reached, do not explore further from this node
            if current_depth >= depth_limit:
                continue

            for next_state in self.get_neighbors(current_state):
                # In DLS, we are interested in paths up to a certain depth.
                # We might visit states multiple times if different paths lead to them.
                # However, for efficiency and to avoid infinite loops in a finite-state space
                # with repeated states, a simple visited set for the current search iteration is useful.
                # For finding a guaranteed shortest path with DFS-like behavior, Iterative Deepening DFS is better.
                if next_state not in visited: # This visited check is simplified for demonstration of DLS structure
                    visited.add(next_state)
                    stack.append((next_state, path + [next_state], current_depth + 1))

        return None, -1 # No solution found within the depth limit

def print_board(board_tuple):
    for row in board_tuple:
        print(list(row))
    print()

# --- Example Usage for 8-Puzzle DLS ---

# Example initial and goal states (represented as tuple of tuples for hashability)
initial_state_dls = (
    (1, 2, 3),
    (0, 4, 6),
    (7, 5, 8)
)

goal_state_dls = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

print("Initial State for DLS:")
print_board(initial_state_dls)

print("Goal State for DLS:")
print_board(goal_state_dls)

puzzle_solver = EightPuzzleDLS(initial_state_dls, goal_state_dls)

# Set a depth limit for the search
depth_limit_value = 5 # Adjust this to test different depths

print(f"\nAttempting DLS with depth limit: {depth_limit_value}")
solution_path_dls, final_depth = puzzle_solver.dls(depth_limit_value)

if solution_path_dls:
    print(f"\nSolution found at depth {final_depth} within the limit!")
    print("Path:")
    for i, state_step in enumerate(solution_path_dls):
        print(f"Step {i} (Depth {i}):")
        print_board(state_step)
else:
    print(f"\nNo solution found within depth limit {depth_limit_value}.")
    print("Try increasing the depth limit, or consider if the puzzle is solvable.")

Initial State for DLS:
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]

Goal State for DLS:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


Attempting DLS with depth limit: 5

Solution found at depth 3 within the limit!
Path:
Step 0 (Depth 0):
[1, 2, 3]
[0, 4, 6]
[7, 5, 8]

Step 1 (Depth 1):
[1, 2, 3]
[4, 0, 6]
[7, 5, 8]

Step 2 (Depth 2):
[1, 2, 3]
[4, 5, 6]
[7, 0, 8]

Step 3 (Depth 3):
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

